# T10 FULL — XRF inference on real spectra

Run **deterministic** inference (one forward pass, dropout disabled) or **Bayesian-style Monte Carlo dropout** inference (multiple stochastic forward passes) using the same fine-tuned T10 FULL model.

The notebook retains the original T10 architecture, real-spectrum normalization, elemental ordering, and `.npz` cache format. It does **not** compare models or apply Pb/Ti/Ca filters.

> MC dropout provides an approximate predictive uncertainty estimate; it is not exact Bayesian posterior inference.

**Inputs:** `best_model_ft_T10_FULL.pth` and JSON spectra in `real_json2/` (each with a 2048-channel `counts` array). **Output:** a cache under `MC_dropout_inference/`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# test spectra path
BASE_DIR = Path.cwd().resolve()
REAL_DIR = BASE_DIR / "real_json2"

#fine tuned model path
T10_MODEL_PATH = BASE_DIR / "best_model_ft_T10_FULL.pth"

#output cache path
CACHE_DIR = BASE_DIR / "MC_dropout_inference"

# "bayesian": Monte Carlo dropout, with MC_SAMPLES stochastic passes.
# "deterministic": one pass with dropout disabled (model.eval()).
INFERENCE_MODE = "bayesian"  # Change to "deterministic" if needed.
MC_SAMPLES = 40              # MC dropout steps
BATCH_SIZE = 200
STORE_SPECTRA = True
LOAD_EXISTING_CACHE = False  # True: load the saved cache instead of running inference.

#elements included in the training of the model
FIXED_ELEMENTS = [
    'Al', 'As', 'Ba', 'Ca', 'Cd', 'Co', 'Cr', 'Cu', 'Fe', 'Hg',
    'K', 'Mg', 'Mn', 'P', 'Pb', 'S', 'Sb', 'Si', 'Sn', 'Ti', 'Zn'
]
elements = FIXED_ELEMENTS

if INFERENCE_MODE not in {"bayesian", "deterministic"}:
    raise ValueError("INFERENCE_MODE must be 'bayesian' or 'deterministic'.")
N_DRAWS = MC_SAMPLES if INFERENCE_MODE == "bayesian" else 1
CACHE_PATH = CACHE_DIR / f"t10_full_{INFERENCE_MODE}_cache.npz"

device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("Device:", device, "| Mode:", INFERENCE_MODE, "| Draws:", N_DRAWS)


## Real XRF dataset


In [ ]:
def extract_real_name(path):
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    labels = d.get("labels", {})
    return str(
        labels.get("title")
        or labels.get("name")
        or labels.get("sample_name")
        or d.get("spectrum_id")
        or path.stem
    )

class RealXRFForMC(Dataset):
    def __init__(self, paths, elements, normalize="max", add_org=True):
        self.paths = list(paths)
        self.elements = list(elements)
        self.elem_to_idx = {e:i for i,e in enumerate(elements)}
        self.normalize = normalize
        self.add_org = add_org
        self.n_elems = len(elements) + 1

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with open(self.paths[idx], "r", encoding="utf-8") as f:
            d = json.load(f)

        counts = np.asarray(d["counts"], dtype=np.float32)
        if self.normalize == "max":
            m = float(counts.max())
            if m > 0:
                counts = counts / m

        labels = d.get("labels", {})
        lp = np.asarray(labels.get("layer_present", [1,0,1]), dtype=np.float32)
        organic = np.asarray(labels.get("organic_frac", [0,0,0]), dtype=np.float32)

        comp = np.zeros((3, self.n_elems), dtype=np.float32)
        comp_dict = labels.get("composition", {})

        for li in range(3):
            # Supports both layer1 and layer_1
            layer = comp_dict.get(f"layer{li+1}", comp_dict.get(f"layer_{li+1}", {}))
            for el, val in layer.items():
                if el in self.elem_to_idx:
                    comp[li, self.elem_to_idx[el]] = float(val)
            if self.add_org:
                comp[li, len(self.elements)] = float(organic[li]) if li < len(organic) else 0.0

        x = torch.from_numpy(counts).unsqueeze(0)
        lp = torch.from_numpy(lp)
        # Placeholder thickness target; real measurements do not provide ground-truth thickness.
        dummy_th = torch.zeros(3, dtype=torch.float32)
        comp = torch.from_numpy(comp)
        return x, lp, dummy_th, comp



In [ ]:
real_paths = sorted(REAL_DIR.glob("*.json"))
if not real_paths:
    raise FileNotFoundError(f"No JSON spectra found in {REAL_DIR}")

real_names = [extract_real_name(path) for path in real_paths]
real_ds = RealXRFForMC(real_paths, elements, normalize="max", add_org=True)
real_loader = DataLoader(
    real_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=torch.cuda.is_available()
)
print("Real spectra:", len(real_ds))


## NN Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class T10XRFMultilayerNetLocalComp(nn.Module):
    """
    T10 FULL architecture.

    Encoder HPO fixed:
        1 -> 64 -> 128 -> 256 -> 256
        kernel=7, stride=2

    Local composition branch T10:
        pool=8
        local_kernel=7
        local_channels=256 -> 128
        shared_hidden=256
        local_dropout=0.10

    Output:
        presence2_logit : [B]
        thickness_log   : [B, 3]
        comp_logits     : [B, 3, len(elements)+1]
    """
    def __init__(
        self,
        n_channels=1,
        spec_len=2048,
        elements=None,
        latent_dim=256,
        comp_pool_bins=8,
        comp_hidden=256,
        local_kernel=7,
        local_channels=256,
        local_dropout=0.10,
        drop_conv=0.05,
        drop_latent=0.10,
        drop_head=0.00,
    ):
        super().__init__()
        if elements is None:
            elements = []

        self.elements = list(elements)
        self.n_layers = 3
        self.n_elems = len(elements) + 1
        self.latent_dim = 256
        self.comp_pool_bins = int(comp_pool_bins)
        self.local_kernel = int(local_kernel)
        self.local_channels = int(local_channels)
        self.local_dropout = float(local_dropout)

        self.do_conv = nn.Dropout1d(p=drop_conv) if drop_conv > 0 else nn.Identity()
        self.do_latent = nn.Dropout(p=drop_latent) if drop_latent > 0 else nn.Identity()
        self.do_head = nn.Dropout(p=drop_head) if drop_head > 0 else nn.Identity()

        # T10/HPO fixed encoder: base_channels=64, depth=4, k=7, all stride=2
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,

            nn.Conv1d(64, 128, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,

            nn.Conv1d(128, 256, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,

            nn.Conv1d(256, 256, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            self.do_conv,
        )

        self.head_presence2 = nn.Linear(256, 1)
        self.head_thickness_log = nn.Linear(256, self.n_layers)

        pad_local = self.local_kernel // 2
        local_hidden_2 = max(self.local_channels // 2, 1)

        comp_pre_layers = [
            nn.Conv1d(256, self.local_channels, kernel_size=self.local_kernel, padding=pad_local),
            nn.ReLU(inplace=True),
        ]
        if self.local_dropout > 0:
            comp_pre_layers.append(nn.Dropout1d(p=self.local_dropout))

        comp_pre_layers += [
            nn.Conv1d(self.local_channels, local_hidden_2, kernel_size=self.local_kernel, padding=pad_local),
            nn.ReLU(inplace=True),
        ]
        if self.local_dropout > 0:
            comp_pre_layers.append(nn.Dropout1d(p=self.local_dropout))

        self.comp_pre = nn.Sequential(*comp_pre_layers)
        self.comp_pool = nn.AdaptiveAvgPool1d(self.comp_pool_bins)

        comp_in_dim = local_hidden_2 * self.comp_pool_bins + 256
        self.comp_shared = nn.Sequential(
            nn.Linear(comp_in_dim, comp_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p=drop_head) if drop_head > 0 else nn.Identity(),
        )

        self.head_comp_l1 = nn.Linear(comp_hidden, self.n_elems)
        self.head_comp_l2 = nn.Linear(comp_hidden, self.n_elems)
        self.head_comp_l3 = nn.Linear(comp_hidden, self.n_elems)

    def forward(self, x):
        z_map = self.encoder(x)
        z_global = self.do_latent(z_map.mean(dim=-1))
        z_head = self.do_head(z_global)

        presence2_logit = self.head_presence2(z_head).squeeze(-1)
        thickness_log = self.head_thickness_log(z_head)

        z_comp_local = self.comp_pre(z_map)
        z_comp_local = self.comp_pool(z_comp_local).flatten(1)
        z_comp = torch.cat([z_comp_local, z_global], dim=1)
        z_comp = self.comp_shared(z_comp)

        comp_logits = torch.stack([
            self.head_comp_l1(z_comp),
            self.head_comp_l2(z_comp),
            self.head_comp_l3(z_comp),
        ], dim=1)

        return {
            "z_global": z_global,
            "z_map": z_map,
            "presence2_logit": presence2_logit,
            "thickness_log": thickness_log,
            "comp_logits": comp_logits,
        }


## Cache sve and loading
`p2_draws` has shape `(N, draws)` and `comp_draws` has shape `(N, 3, E+1, draws)`. For deterministic inference, `draws = 1`. The cache also stores predicted log areal densities, the input spectra (optionally), sample names, and available ground-truth labels.

In [ ]:
def save_mc_cache_npz(cache, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays, meta = {}, {}
    for key, value in cache.items():
        if isinstance(value, np.ndarray):
            arrays[key] = value
        else:
            meta[key] = value
    if "elements_fixed" in cache and not isinstance(cache["elements_fixed"], np.ndarray):
        arrays["elements_fixed_arr"] = np.asarray(cache["elements_fixed"], dtype=object)
        meta.pop("elements_fixed", None)
    arrays["__meta__"] = np.array([meta], dtype=object)
    np.savez_compressed(path, **arrays)
    print("Saved cache:", path)
    return path


def load_mc_cache_npz(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Cache not found: {path}")
    cache = {}
    with np.load(path, allow_pickle=True) as data:
        for key in data.files:
            if key == "__meta__":
                obj = data[key]
                metadata = obj.item() if obj.size == 1 else obj.reshape(-1)[0]
                if isinstance(metadata, dict):
                    cache.update(metadata)
            elif key == "elements_fixed_arr":
                cache["elements_fixed"] = data[key].tolist()
            else:
                cache[key] = data[key]
    for key in ("mc_samples", "org_index", "M", "comp_topk"):
        if key in cache:
            cache[key] = int(cache[key])
    return cache


## MC or deterministic inference

In [ ]:
def configure_inference_mode(model, mode):
    model.eval()
    if mode == "bayesian":
        for module in model.modules():
            if isinstance(module, (nn.Dropout, nn.Dropout1d, nn.Dropout2d, nn.Dropout3d)):
                module.train()


@torch.no_grad()
def run_t10_inference(model, loader, device, mode, n_draws, elements_fixed, real_names, store_spectra=True):
    configure_inference_mode(model, mode)
    N = len(loader.dataset)
    n_comp = len(elements_fixed) + 1
    p2_draws = np.empty((N, n_draws), dtype=np.float32)
    comp_draws = np.empty((N, 3, n_comp, n_draws), dtype=np.float32)
    thickness_log_draws = np.empty((N, 3, n_draws), dtype=np.float32)
    layer_present_all = np.empty((N, 3), dtype=np.float32)
    comp_true_all = np.empty((N, 3, n_comp), dtype=np.float32)
    spectra_all = np.empty((N, 2048), dtype=np.float32) if store_spectra else None

    offset = 0
    for batch in tqdm(loader, desc=f"T10 {mode} inference", unit="batch"):
        x = batch[0].to(device, non_blocking=True)
        B = x.shape[0]
        i0, i1 = offset, offset + B
        layer_present_all[i0:i1] = batch[1].cpu().numpy()
        comp_true_all[i0:i1] = batch[3].cpu().numpy()
        if store_spectra:
            spectra_all[i0:i1] = x[:, 0, :].cpu().numpy()

        for draw in range(n_draws):
            out = model(x)
            p2_draws[i0:i1, draw] = torch.sigmoid(out["presence2_logit"]).reshape(-1).cpu().numpy()
            comp_draws[i0:i1, :, :, draw] = torch.softmax(out["comp_logits"], dim=-1).cpu().numpy()
            thickness_log_draws[i0:i1, :, draw] = out["thickness_log"].cpu().numpy()
        offset = i1

    if offset != N:
        raise RuntimeError(f"Incomplete cache: processed {offset}/{N} spectra")
    if not (np.isfinite(p2_draws).all() and np.isfinite(comp_draws).all()
            and np.isfinite(thickness_log_draws).all()):
        raise RuntimeError("Non-finite inference outputs")
    if not np.allclose(comp_draws.sum(axis=2), 1.0, atol=1e-4):
        raise RuntimeError("Composition probabilities do not sum to one")

    cache = {
        "p2_draws": p2_draws,
        "comp_draws": comp_draws,
        "thickness_log_draws": thickness_log_draws,
        "layer_present": layer_present_all,
        "comp_true": comp_true_all,
        "elements_fixed": list(elements_fixed),
        "org_index": len(elements_fixed),
        "mc_samples": n_draws,
        "M": n_draws,
        "real_names": np.asarray(real_names, dtype=object),
        "inference_mode": mode,
    }
    if store_spectra:
        cache["spectra"] = spectra_all
    return cache


## Run or load the cache

In [ ]:
if LOAD_EXISTING_CACHE:
    cache_t10 = load_mc_cache_npz(CACHE_PATH)
else:
    if not T10_MODEL_PATH.exists():
        raise FileNotFoundError(f"T10 checkpoint not found: {T10_MODEL_PATH}")
    checkpoint = torch.load(T10_MODEL_PATH, map_location=device, weights_only=False)
    state = checkpoint["model_state"] if isinstance(checkpoint, dict) and "model_state" in checkpoint else checkpoint
    model_t10 = T10XRFMultilayerNetLocalComp(
        n_channels=1, spec_len=2048, elements=elements,
        latent_dim=256, comp_pool_bins=8, comp_hidden=256,
        local_kernel=7, local_channels=256, local_dropout=0.10,
        drop_conv=0.05, drop_latent=0.10, drop_head=0.00,
    ).to(device)
    model_t10.load_state_dict(state, strict=True)
    cache_t10 = run_t10_inference(
        model_t10, real_loader, device, INFERENCE_MODE,
        N_DRAWS, elements, real_names, store_spectra=STORE_SPECTRA
    )
    save_mc_cache_npz(cache_t10, CACHE_PATH)

print("Cache path:", CACHE_PATH)
print("Presence probabilities:", np.asarray(cache_t10["p2_draws"]).shape)
print("Composition probabilities:", np.asarray(cache_t10["comp_draws"]).shape)
print("Log areal density:", np.asarray(cache_t10["thickness_log_draws"]).shape)
